# client

> Client for interacting with the Fewsats API

In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.utils import *
import os
import httpx
from typing import Dict, Any, List
from time import time, sleep
import json
from httpx import HTTPError

In [ ]:
#| hide 
from dotenv import load_dotenv
from fastcore.test import *

In [ ]:
#| hide
load_dotenv()

True

The `Fewsats` class handles authentication and provides the foundation for our API interactions.

In [ ]:
#| export
class Fewsats:
    "Client for interacting with the Fewsats API"
    def __init__(self,
                 api_key: str = None, # The API key for the Fewsats account
                 base_url: str = "https://hub-5n97k.ondigitalocean.app"): # The Fewsats API base URL
        self.api_key = api_key or os.environ.get("FEWSATS_API_KEY")
        if not self.api_key:
            raise ValueError("The api_key client option must be set either by passing api_key to the client or by setting the FEWSATS_API_KEY environment variable")
        self.base_url = base_url
        self._httpx_client = httpx.Client()
        self._httpx_client.headers.update({"Authorization": f"Token {self.api_key}"})


In [ ]:
k = os.getenv("FEWSATS_API_KEY")
fs = Fewsats(api_key=k)
# k = os.getenv("FEWSATS_LOCAL_API_KEY")
# fs = Fewsats(api_key=k, base_url="http://localhost:8000")

test_eq(fs.api_key, k)
test_eq(fs._httpx_client.headers["Authorization"], f"Token {k}")

## Methods

In [ ]:
#| export
@patch
def _request(self: Fewsats, 
             method: str, # The HTTP method to use
             path: str, # The path to request
             timeout: int = 10, # Timeout for the request in s
             **kwargs) -> Dict[str, Any]:
    "Makes an authenticated request to Fewsats API"
    url = f"{self.base_url}/{path}"
    return  self._httpx_client.request(method, url, timeout=timeout, **kwargs)

In [ ]:
r  = fs._request("GET", "v0/users/me")
test_eq(r.status_code, 200)

Let's use a helper function to process the response.

In [ ]:
# #| export
# def _process_response(r):
#     try:
#         r.raise_for_status()
#         return r.json()
#     except (json.JSONDecodeError, HTTPError) as e:
#         if isinstance(e, json.JSONDecodeError):
#             error = str(e)
#         else:
#             reason = e.reason
#             details = e.details  # Includes additional info like headers
#             error = f"{reason}: {details}"
#         raise Exception(f"Status: {e.status_code}, Error: {error}")
#     except Exception as e:
#         raise Exception(str(e))

### User Info

In [ ]:
#| export

@patch
def me(self: Fewsats):
    "Retrieve the user's info."
    return self._request("GET", "v0/users/me")


In [ ]:
r = fs.me()
r.status_code, r.json()

(200,
 {'name': 'Fewsats',
  'last_name': 'Tester',
  'email': 'test@fewsats.com',
  'billing_info': None,
  'id': 15,
  'created_at': '2024-12-18T18:19:00.531Z',
  'webhook_url': 'https://example.com'})

### Balance 

In [ ]:
#| export 

@patch
def balance(self: Fewsats):
    "Retrieve the balance of the user's wallet."
    return self._request("GET", "v0/wallets")


In [ ]:
r = fs.balance()
r.status_code, r.json()

(200, [{'id': 15, 'balance': 2373, 'currency': 'usd'}])

### Payment Methods

Retrieve the user's payment methods. Useful for checking which card will be used for purchases.

In [ ]:
#| export
@patch
def payment_methods(self: Fewsats) -> List[Dict[str, Any]]:
    "Retrieve the user's payment methods, raises an exception for error status codes."
    return self._request("GET", "v0/stripe/payment-methods")


In [ ]:
r = fs.payment_methods()
payment_methods = r.json()
r.status_code, payment_methods

(200,
 [{'id': 5,
   'last4': '4242',
   'brand': 'Visa',
   'exp_month': 12,
   'exp_year': 2034,
   'is_default': True}])

In [ ]:
assert isinstance(payment_methods, list)

### Preview a Purchase

Preview the resulting state of a purchase. Useful, for example, to check if a CC charge is needed or the purchase will use the balance.

In [ ]:
#| export

@patch
def _preview_payment(self: Fewsats,
                    amount: str): # The amount in USD cents
    "Simulates a purchase, raises an exception for error status codes."
    assert amount.isdigit()
    return self._request("POST", "v0/l402/preview/purchase/amount", json={"amount_usd": amount})


In [ ]:
r = fs._preview_payment(amount="300") # 3.00 USD
preview = r.json()
r.status_code = preview

### Create offers

How to use the client to generate L402 offers

In [ ]:
#| export
@patch
def create_offers(self:Fewsats,
                 offers:List[Dict[str,Any]], # List of offer objects following OfferCreateV0 schema
) -> dict:
    "Create offers for L402 payment server"
    return self._request("POST", "v0/l402/offers", json={"offers": offers})

In [ ]:
test_offers = [{
    "offer_id": "test_offer_2",
    "amount": 1,
    "currency": "USD",
    "description": "Test offer",
    "title": "Test Package",
    "tye": "top-up",
    "payment_methods": ["lightning", "credit_card"]
}]

r = fs.create_offers(test_offers)
l402_offers = r.json()
r.status_code, l402_offers

(200,
 {'offers': [{'offer_id': 'test_offer_2',
    'amount': 1,
    'currency': 'USD',
    'description': 'Test offer',
    'title': 'Test Package',
    'payment_methods': ['lightning', 'credit_card'],
    'type': 'one-off'}],
  'payment_context_token': '8eff76bf-1d99-4dc2-b6b7-537ef2b62998',
  'payment_request_url': 'https://api.fewsats.com/v0/l402/payment-request',
  'version': '0.2.2'})

### Get Payment Details

In [ ]:
#| export
@patch
def get_payment_details(self:Fewsats,
                       payment_request_url:str,
                       offer_id:str,
                       payment_method:str,
                       payment_context_token:str,
                       ) -> dict:
    data = {"offer_id": offer_id, "payment_method": payment_method, "payment_context_token": payment_context_token}
    return httpx.post(payment_request_url, json=data)


In [ ]:
r = fs.get_payment_details(l402_offers["payment_request_url"], l402_offers["offers"][0]["offer_id"], "lightning", l402_offers["payment_context_token"])
payment_details = r.json()
ln_invoice = payment_details["payment_request"]['lightning_invoice']
r.status_code, payment_details

(200,
 {'expires_at': '2025-02-27T06:38:36.838131+00:00',
  'offer_id': 'test_offer_2',
  'payment_request': {'lightning_invoice': 'lnbc110n1pnuqqp6pp58ecgelkk7x7f3k785hhcpa6pnmq03w60302vne3wt2v7xltm35kqdq523jhxapq2pskx6mpvajscqzpgxqrzpjrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp5tup74y2nsk5q8e03cae0apdr28x2m3eav94lurywpl02j7sv5p3s9qxpqysgqal7sefttee3xlh3x87q7fvqnwaa56fzd8q8w5tpa62u9vywutrmylxxmdfez82d5saj89vjz30y79prxuedt3a6aylrk32lt0dvtlvcqkx0a8r'},
  'version': '0.2.2'})

### Get Payment Status


In [ ]:
#| export
@patch
def get_payment_status(self:Fewsats,
                       payment_context_token:str,
                       ) -> dict:
    return self._request("GET", f"v0/l402/payment-status?payment_context_token={payment_context_token}")

In [ ]:

r = fs.get_payment_status(l402_offers["payment_context_token"])
r.status_code, r.json()

(200,
 {'payment_context_token': '8eff76bf-1d99-4dc2-b6b7-537ef2b62998',
  'status': 'pending',
  'offer_id': None,
  'paid_at': None,
  'amount': None,
  'currency': None})

In [ ]:
#| export
@patch
def set_webhook(self:Fewsats,
                       webhook_url:str,
                       ) -> dict:
    return self._request("POST", f"v0/users/webhook/set", json={"webhook_url": webhook_url})

In [ ]:

r = fs.set_webhook("https://example.com")
r, r.json()

(<Response [200 OK]>,
 {'name': 'Fewsats',
  'last_name': 'Tester',
  'email': 'test@fewsats.com',
  'billing_info': None,
  'id': 15,
  'created_at': '2024-12-18T18:19:00.531Z',
  'webhook_url': 'https://example.com'})

### Pay Lightning Invoice

In [ ]:
#| export

@patch
def pay_lightning(self: Fewsats, 
                  invoice: str, # lightning invoice
                  amount: int, # amount in cents
                  currency: str = "USD", # currency
                  description: str = "" ): # description of the payment 
    "Pay for a lightning invoice"
    data = {
        "invoice": invoice,
        "amount": amount,
        "currency": currency,
        "description": description
    }
    return self._request("POST", "v0/l402/purchases/lightning", json=data)

In [ ]:
r = fs.pay_lightning(invoice=ln_invoice,
                     description="fewsats webhook trial", amount=1)
lightning_payment = r.json()
r.status_code, lightning_payment

(200,
 {'id': 313,
  'created_at': '2025-02-27T06:03:40.805Z',
  'status': 'success',
  'payment_request_url': '',
  'payment_context_token': '',
  'invoice': 'lnbc110n1pnuqqp6pp58ecgelkk7x7f3k785hhcpa6pnmq03w60302vne3wt2v7xltm35kqdq523jhxapq2pskx6mpvajscqzpgxqrzpjrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp5tup74y2nsk5q8e03cae0apdr28x2m3eav94lurywpl02j7sv5p3s9qxpqysgqal7sefttee3xlh3x87q7fvqnwaa56fzd8q8w5tpa62u9vywutrmylxxmdfez82d5saj89vjz30y79prxuedt3a6aylrk32lt0dvtlvcqkx0a8r',
  'preimage': '93de3658feeba62f6156001ac0a2a16e7bfe2b41f64417beea3e2b93b3b904c4',
  'amount': 1,
  'currency': 'usd',
  'payment_method': 'lightning',
  'title': '',
  'description': 'fewsats webhook trial',
  'type': ''})

In [ ]:
fs.get_payment_status(l402_offers["payment_context_token"]).json()

{'payment_context_token': '8eff76bf-1d99-4dc2-b6b7-537ef2b62998',
 'status': 'pending',
 'offer_id': None,
 'paid_at': None,
 'amount': None,
 'currency': None}

### Pay 

The pay method pays for a specific offer. The user is not required to fetch the payment details beforehand. It is asynchronous and returns the `payment_id` and `status`. Using the `payment_id` we can check the status of the payment.

In [ ]:
#| export


@patch
def pay_offer(self:Fewsats,
        l402_offer: Dict, # a dictionary containing the response of an L402 endpoint
        payment_method:str = '', # preferred payment method (optional)
) -> dict: # payment status response
    """Pays an L402 response. Fewsats will choose payment method if left blank.
    If multiple offers are passed, the first one will be chosen.

    This method is not recommended for LLMs as they struggle with complex types like Dicts during
    function calling. This method should be used by a higher-level abstraction SDK that is in 
    turn exposed to the LLM, for example, with the `as_tools()` paradigm.

    Returns payment status response"""
    data = {"payment_method": payment_method, **l402_offer} if payment_method else l402_offer
    return self._request("POST", "v0/l402/purchases/from-offer", timeout=20, json=data)

In [ ]:
r = fs.pay_offer(l402_offers)
payment_response = r.json() if r.is_success else r.text
r.status_code, payment_response

(400,
 '{"detail": "Invalid payment request received. Could not get payment details. {\\"detail\\": \\"Invalid payment request received. Payment context token \'8eff76bf-1d99-4dc2-b6b7-537ef2b62998\' already used\\"}"}')

After the stripe payment settles, the status will be updated to `success`.

### Payment Info

We can check the status of a payment as follows:

In [ ]:
#| export
@patch
def payment_info(self:Fewsats,
                  pid:str): # purchase id
    "Retrieve the details of a payment."
    return self._request("GET", f"v0/l402/purchases/{pid}")

In [ ]:
fs.payment_info('2')

<Response [404 Not Found]>

### Wait for settlement

For convenience we can use the method `wait_for_settlement` to wait for the payment to settle.

In [ ]:
#| export
@patch
def wait_for_settlement(self:Fewsats,
                        pid:str, # purchase id
                        max_interval:int=120, # maximum interval between checks in seconds
                        max_wait:int=600): # maximum total wait time in seconds
    "Wait for payment settlement with exponential backoff"
    start,wait = time(),1
    while time() - start < max_wait:
        r = self.payment_info(pid)
        r.raise_for_status()
        status = r['status']
        if status == 'success': return r
        if status == 'failed': raise ValueError(f"Payment {pid} failed")
        sleep(min(wait, max_interval))
        wait *= 2
    raise TimeoutError(f"Payment {pid} did not settle within {max_wait} seconds. Final status: {status}")

In [ ]:
#| skip
try:
    r = fs.wait_for_settlement('223') # test payment already settled
except Exception as e:
    print(e)

Client error '404 Not Found' for url 'http://localhost:8000/v0/l402/purchases/223'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404


## As tools

In [ ]:
#| export

@patch
def as_tools(self:Fewsats):
    "Return list of available tools for AI agents"
    return [
        self.me,
        self.balance,
        self.payment_methods,
        self.pay_lightning,
        self.payment_info,
    ]

In [ ]:
fs.as_tools()

[<bound method Client.me of <__main__.Client object>>,
 <bound method Client.balance of <__main__.Client object>>,
 <bound method Client.payment_methods of <__main__.Client object>>,
 <bound method Client.pay of <__main__.Client object>>,
 <bound method Client.payment_info of <__main__.Client object>>]

Both the preview and purchase methods automatically use the default payment method if a charge is needed. This client provides a straightforward way to interact with the Fewsats API, making it easy for developers to integrate Fewsats functionality into their applications.

## Agent Demo

We will use [Claudette](https://claudette.answer.ai/) to demonstrate how to pay for content using the Fewsats API.

In [ ]:
from claudette import Chat, models

In [ ]:
model = models[1]
model

'claude-3-5-sonnet-20240620'

In [ ]:
fs.balance()

[{'id': 15, 'balance': 4959, 'currency': 'usd'}]

In [ ]:
chat = Chat(model, sp='You are a helpful assistant that can pay offers.', tools=fs.as_tools())
pr = f"Could you pay the cheapest offer using lightning {ofs}?"
r = chat.toolloop(pr, trace_func=print)
r

Message(id='msg_01RCWZnqoZXUsQJgWpUwqZ7Z', content=[TextBlock(text='Certainly! I\'d be happy to help you pay for the cheapest offer using Lightning. Let\'s analyze the offers and proceed with the payment.\n\nFrom the information you\'ve provided, the cheapest offer is:\n\n- Amount: 1 cent (USD)\n- Balance: 1 credit\n- Offer ID: offer_c668e0c0\n- Title: "1 Credit Package"\n- Type: top-up\n- Payment method: Lightning\n\nNow, let\'s use the `pay` function to process this payment. We\'ll need to use the information from the cheapest offer and the payment context you\'ve provided.', type='text'), ToolUseBlock(id='toolu_01MCXoRCLnskTnYTEwHDVQyw', input={'purl': 'https://stock.l402.org/l402/payment-request', 'pct': 'edb53dec-28f5-4cbb-924a-20e9003c20e1', 'amount': 1, 'balance': 1, 'currency': 'USD', 'description': 'Purchase 1 credit for API access', 'offer_id': 'offer_c668e0c0', 'payment_methods': ['lightning'], 'title': '1 Credit Package', 'type': 'top-up'}, name='pay', type='tool_use')], mo

Great news! The payment for the cheapest offer has been successfully processed. Here's a summary of the transaction:

1. Payment Status: Success
2. Amount Paid: 1 cent (USD)
3. Payment Method: Lightning
4. Title: 1 Credit Package
5. Description: Purchase 1 credit for API access
6. Type: Top-up
7. Transaction ID: 264
8. Created At: 2025-01-25T19:52:55.228Z (Note: This appears to be a future date, which might be due to a system time discrepancy)

The payment has been completed, and you should now have 1 credit added to your API access. Is there anything else you would like me to help you with regarding this transaction or any other matters?

<details>

- id: `msg_016QxjwdSQchw5Vmyqqu23ey`
- content: `[{'text': "Great news! The payment for the cheapest offer has been successfully processed. Here's a summary of the transaction:\n\n1. Payment Status: Success\n2. Amount Paid: 1 cent (USD)\n3. Payment Method: Lightning\n4. Title: 1 Credit Package\n5. Description: Purchase 1 credit for API access\n6. Type: Top-up\n7. Transaction ID: 264\n8. Created At: 2025-01-25T19:52:55.228Z (Note: This appears to be a future date, which might be due to a system time discrepancy)\n\nThe payment has been completed, and you should now have 1 credit added to your API access. Is there anything else you would like me to help you with regarding this transaction or any other matters?", 'type': 'text'}]`
- model: `claude-3-5-sonnet-20240620`
- role: `assistant`
- stop_reason: `end_turn`
- stop_sequence: `None`
- type: `message`
- usage: `{'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 2146, 'output_tokens': 181}`

</details>

We can see in the chat history to see that the agent correctlye filled the required information for the payment.

The payment balance has also decreased as expected.

In [ ]:
fs.balance(), chat.h

([{'id': 15, 'balance': 4958, 'currency': 'usd'}],
 [{'role': 'user',
   'content': [{'type': 'text',
     'text': "Could you pay the cheapest offer using lightning {'offers': [{'amount': 1, 'balance': 1, 'currency': 'USD', 'description': 'Purchase 1 credit for API access', 'offer_id': 'offer_c668e0c0', 'payment_methods': ['lightning'], 'title': '1 Credit Package', 'type': 'top-up'}, {'amount': 100, 'balance': 120, 'currency': 'USD', 'description': 'Purchase 120 credits for API access', 'offer_id': 'offer_97bf23f7', 'payment_methods': ['lightning', 'coinbase_commerce'], 'title': '120 Credits Package', 'type': 'top-up'}, {'amount': 499, 'balance': 750, 'currency': 'USD', 'description': 'Purchase 750 credits for API access', 'offer_id': 'offer_a896b13c', 'payment_methods': ['lightning', 'coinbase_commerce', 'credit_card'], 'title': '750 Credits Package', 'type': 'top-up'}], 'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1', 'payment_request_url': 'https://stock.l402.org/l40

In [ ]:
#|hide
from nbdev.doclinks import nbdev_export
nbdev_export()